In [1]:
"""Phase 5 — Grouped DeepMMNet, multi-rate, on `systematic-720`.

    conda run -n exadigit python run_phase5.py

Shared task spec / data / metrics / report: see `fmu2ml/surrogate/phase_common.py`.

ARCHITECTURE (`GroupedDeepMMNet` — natively multi-rate)
    The library's grouped model IS the multi-rate backbone MR-MIONet was
    built from: per-branch u/y LSTM encoders, one shared Fourier trunk, and
    per-group decoder heads (standard or skip). Phase 5 is therefore the
    direct, in-library ancestor of run19b — the honest "what does the base
    grouped model score before the run-series' custom additions
    (future-forcing branch, attention fusion, GatedDeltaHead, beta-NLL,
    per-head checkpointing)."

TRAINING
    Two-phase protocol (round-robin per-head warmup, then joint fine-tuning
    with differential LRs) driven by `phase1_epochs` / `phase2_epochs`, NOT
    `max_epochs`.
"""

'Phase 5 — Grouped DeepMMNet, multi-rate, on `systematic-new`.\n\n    conda run -n exadigit python run_phase5.py\n\nShared task spec / data / metrics / report: see `phase_common.py`.\n\nARCHITECTURE (`GroupedDeepMMNet` — natively multi-rate)\n    The library\'s grouped model IS the multi-rate backbone MR-MIONet was\n    built from: per-branch u/y LSTM encoders, one shared Fourier trunk, and\n    per-group decoder heads (standard or skip). Phase 5 is therefore the\n    direct, in-library ancestor of run19b — the honest "what does the base\n    grouped model score before the run-series\' custom additions\n    (future-forcing branch, attention fusion, GatedDeltaHead, beta-NLL,\n    per-head checkpointing)."\n\nTRAINING\n    Two-phase protocol (round-robin per-head warmup, then joint fine-tuning\n    with differential LRs) driven by `phase1_epochs` / `phase2_epochs`, NOT\n    `max_epochs`.\n'

In [ ]:
# 1  Imports, seed, run registry (bump RUN_ID every run; NEVER reuse one)
import os, sys, json, time, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Use THIS repo's fmu2ml (<repo>/surrogate_models/..), whatever the kernel's
# working directory, PYTHONPATH, or an earlier import in this kernel say.
_nb_file = globals().get("__vsc_ipynb_file__")      # notebook path under VS Code
NB_DIR = os.path.dirname(os.path.abspath(_nb_file)) if _nb_file else os.getcwd()
REPO_ROOT = os.path.dirname(NB_DIR)
os.chdir(NB_DIR)                                    # run outputs -> surrogate_models/runs/
for _m in [m for m in sys.modules if m == "fmu2ml" or m.startswith("fmu2ml.")]:
    del sys.modules[_m]                             # forget an fmu2ml imported from elsewhere
sys.path.insert(0, REPO_ROOT)
import fmu2ml.surrogate as surrogate
print("fmu2ml:", os.path.dirname(surrogate.__file__))
assert surrogate.__file__.startswith(os.path.join(REPO_ROOT, "fmu2ml", "")), (
    f"fmu2ml came from {surrogate.__file__}, expected {REPO_ROOT}/fmu2ml "
    f"(the repo above {NB_DIR}; run the kernel from this notebook's folder)")


In [ ]:

from fmu2ml.surrogate.phase_common import (PhaseRun, task_kwargs,
                                          MAX_EPOCHS, PATIENCE, WEIGHT_DECAY, scaled_lr)
import fmu2ml.surrogate as surrogate


In [ ]:

RUN_ID = "phase5_grouped_mr"

RUN_NOTES = (
    "Phase 5 grouped DeepMMNet trained MULTI-RATE on the 720 h "
    "`systematic-720` data, identical task to run19b and the other phases.\n\n"
    "Architecture: per-branch u/y LSTM encoders + one shared Fourier trunk + "
    "per-group decoder heads (standard/skip per head_types). This is the "
    "in-library ancestor of the MR-MIONet run series: run19b = this backbone "
    "PLUS a future-forcing branch, attention fusion, one learnable "
    "GatedDeltaHead per group, beta-NLL, and per-head best checkpointing. "
    "Phase 5's gap to run19b is exactly the value of those additions.\n\n"
    "Two-phase training: round-robin per-head warmup (phase1_epochs) then "
    "joint fine-tuning with differential LRs (phase2_epochs).\n\n"
    "Read by mean SKILL vs persistence, not mean R²."
)


In [ ]:

config = surrogate.Phase5Config(
    **task_kwargs(),

    #  phase-5 architecture (the federated backbone) ────────────────────────
    u_branch_lstm_hidden=128,
    u_branch_lstm_layers=2,
    u_branch_lstm_dropout=0.3,
    y_branch_lstm_hidden=128,
    y_branch_lstm_layers=2,
    y_branch_lstm_dropout=0.3,
    trunk_n_fourier=8,
    trunk_hidden_sizes=[128, 128],
    basis_dim=256,
    head_hidden_sizes={"G_T": [256, 128], "G_V": [256, 128], "G_p": [256, 128],
                       "G_Vs": [128, 64], "G_ps": [128, 64], "G_W": [128, 64]},

    #  training (two-phase; NOT max_epochs) ─────────────────────────────────
    learning_rate=scaled_lr(1e-3),
    # weight decay and patience are shared by every phase (see
    # phase_common); these are the values the reported runs used.
    weight_decay=WEIGHT_DECAY,
    phase1_epochs=50,
    phase2_epochs=100,
    patience=PATIENCE,
    loss_type="huber",
    huber_delta=0.5,
)

if __name__ == "__main__":
    PhaseRun(run_id=RUN_ID, arch="federated", phase="5",
             config=config, notes=RUN_NOTES).execute()
